In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [2]:
df = pd.read_csv("Crop_recommendation.csv")

df.head()

,Nitrogen,Phosphorus,Potassium,Temperature,Humidity,pH_Value,Rainfall,Crop
0,90,42,43,20.879744,82.002744,6.502985,202.935536,Rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,Rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,Rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,Rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,Rice


In [3]:
print(df.shape)

print("\nMissing Values")
print(df.isnull().sum())

print("\nDuplicate Rows")
print(df.duplicated().sum())

print("\nCrop Classes")
print(df["Crop"].value_counts())

(2200, 8)

Missing Values
Nitrogen       0
Phosphorus     0
Potassium      0
Temperature    0
Humidity       0
pH_Value       0
Rainfall       0
Crop           0
dtype: int64

Duplicate Rows
0

Crop Classes
Crop
Rice           100
Maize          100
ChickPea       100
KidneyBeans    100
PigeonPeas     100
MothBeans      100
MungBean       100
Blackgram      100
Lentil         100
Pomegranate    100
Banana         100
Mango          100
Grapes         100
Watermelon     100
Muskmelon      100
Apple          100
Orange         100
Papaya         100
Coconut        100
Cotton         100
Jute           100
Coffee         100
Name: count, dtype: int64


In [4]:
X = df.drop("Crop", axis=1)
y = df["Crop"]

In [5]:
le = LabelEncoder()

y = le.fit_transform(y)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [7]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [8]:
models = {
    "KNN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ),

    "SVM": SVC(),

    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="mlogloss"
    )
}

In [9]:
results = []

for name, model in models.items():

    model.fit(X_train_scaled, y_train)

    preds = model.predict(X_test_scaled)

    accuracy = accuracy_score(y_test, preds)

    precision = precision_score(
        y_test,
        preds,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        preds,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        preds,
        average="weighted"
    )

    results.append([
        name,
        accuracy,
        precision,
        recall,
        f1
    ])

In [13]:
results_df = pd.DataFrame(
    results,
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ]
)

results_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score
2,Random Forest,0.995455,0.995671,0.995455,0.995452
4,XGBoost,0.993182,0.993506,0.993182,0.993116
3,SVM,0.984091,0.985610,0.984091,0.984038
0,KNN,0.979545,0.980356,0.979545,0.979283
1,Decision Tree,0.979545,0.980598,0.979545,0.979423


In [12]:
best_model = models["Random Forest"]

In [11]:
from sklearn.model_selection import RandomizedSearchCV

params = {
    "n_estimators": [100,200,300,500],
    "max_depth": [5,10,15,20,None],
    "min_samples_split": [2,5,10],
    "min_samples_leaf": [1,2,4]
}

search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    params,
    n_iter=20,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

search.fit(X_train_scaled, y_train)

print(search.best_params_)
print(search.best_score_)


{'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 15}
0.9954545454545454


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score, learning_curve
import matplotlib.pyplot as plt

# Evaluate on training data
train_preds = best_model.predict(X_train_scaled)
train_acc = accuracy_score(y_train, train_preds)
train_prec = precision_score(y_train, train_preds, average='weighted')
train_rec = recall_score(y_train, train_preds, average='weighted')
train_f1 = f1_score(y_train, train_preds, average='weighted')

# Evaluate on test data
test_preds = best_model.predict(X_test_scaled)
test_acc = accuracy_score(y_test, test_preds)
test_prec = precision_score(y_test, test_preds, average='weighted')
test_rec = recall_score(y_test, test_preds, average='weighted')
test_f1 = f1_score(y_test, test_preds, average='weighted')

print('Train -> Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}'.format(train_acc, train_prec, train_rec, train_f1))
print('Test  -> Accuracy: {:.4f}, Precision: {:.4f}, Recall: {:.4f}, F1: {:.4f}'.format(test_acc, test_prec, test_rec, test_f1))

# Cross-validation on training set
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print('CV accuracy (5-fold) on train: mean={:.4f}, std={:.4f}'.format(cv_scores.mean(), cv_scores.std()))

# Learning curve
train_sizes, train_scores, val_scores = learning_curve(best_model, X_train_scaled, y_train, cv=5, scoring='accuracy', train_sizes=np.linspace(0.1,1.0,5), n_jobs=-1)
train_scores_mean = np.mean(train_scores, axis=1)
val_scores_mean = np.mean(val_scores, axis=1)

plt.figure(figsize=(8,5))
plt.plot(train_sizes, train_scores_mean, 'o-', color='r', label='Training score')
plt.plot(train_sizes, val_scores_mean, 'o-', color='g', label='Cross-validation score')
plt.xlabel('Training examples')
plt.ylabel('Accuracy')
plt.title('Learning Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()

# Confusion matrix and classification report on test set
print('\nClassification Report (test):')
print(classification_report(y_test, test_preds))
cm = confusion_matrix(y_test, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(figsize=(10,8), xticks_rotation='vertical')
plt.show()

# Quick overfitting check
if train_acc - test_acc > 0.05:
    print('\nWarning: model may be overfitting (train - test > 0.05)')
else:
    print('\nNo strong evidence of overfitting from these metrics.')

In [ ]:
import joblib

joblib.dump(best_model, "crop_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(le, "label_encoder.pkl")